# Notebook 02: Data Preprocessing & Augmentation Pipeline
**Project:** Deep Learning-Based COVID-19 Detection from Chest X-ray Images Using Explainable AI  
**Student:** Channabasavanna Santosh Pawate (16150425)  

## Objective
Build the preprocessing and augmentation pipeline for ViT-B/16 training.

**Key steps:**
1. Resize to 224×224 (ViT-B/16 requirement)
2. Greyscale → RGB channel replication
3. ImageNet normalisation
4. Training augmentation: flip, rotate, colour jitter, random crop
5. Stratified train/val/test split (70/15/15)

In [ ]:
import os
import shutil
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# ImageNet statistics used for normalisation
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
IMG_SIZE = 224

print(f'PyTorch version: {torch.__version__}')
print(f'Image size: {IMG_SIZE}×{IMG_SIZE}')
print(f'ImageNet mean: {IMAGENET_MEAN}')
print(f'ImageNet std:  {IMAGENET_STD}')

## 1. Transform Pipelines

In [ ]:
# --- Training transforms (with augmentation) ---
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 20, IMG_SIZE + 20)),   # Slightly larger for random crop
    transforms.RandomCrop(IMG_SIZE),                       # Random crop to 224
    transforms.RandomHorizontalFlip(p=0.5),               # Horizontal flip
    transforms.RandomRotation(degrees=15),                 # ±15° rotation
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.1
    ),                                                     # Colour jitter
    transforms.Grayscale(num_output_channels=3),          # Ensure 3-channel
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# --- Validation/Test transforms (deterministic) ---
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

print('Train transforms:')
print(train_transform)
print('\nVal/Test transforms:')
print(val_transform)

## 2. Dataset Class

In [ ]:
class ChestXrayDataset(Dataset):
    """Custom dataset for chest X-ray classification.
    
    Expects directory structure:
        root/
          Normal/
          COVID-19/
          Viral Pneumonia/
    """
    
    CLASSES = ['Normal', 'COVID-19', 'Viral Pneumonia']
    CLASS2IDX = {c: i for i, c in enumerate(CLASSES)}
    
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.samples = []  # list of (path, label_idx)
        self._load_samples()
    
    def _load_samples(self):
        for cls in self.CLASSES:
            cls_dir = os.path.join(self.root_dir, cls)
            if not os.path.exists(cls_dir):
                continue
            for fname in os.listdir(cls_dir):
                if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                    self.samples.append((os.path.join(cls_dir, fname),
                                         self.CLASS2IDX[cls]))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label
    
    def get_class_weights(self):
        """Compute class weights for imbalanced dataset."""
        from collections import Counter
        label_counts = Counter(label for _, label in self.samples)
        total = len(self.samples)
        weights = [total / (len(self.CLASSES) * label_counts.get(i, 1))
                   for i in range(len(self.CLASSES))]
        return torch.FloatTensor(weights)


print('ChestXrayDataset class defined.')
print('Classes:', ChestXrayDataset.CLASSES)
print('Label mapping:', ChestXrayDataset.CLASS2IDX)

## 3. DataLoader Factory

In [ ]:
def get_dataloaders(data_root, batch_size=32, num_workers=4):
    """Create train, val, and test DataLoaders.
    
    Args:
        data_root: Path containing 'train/', 'val/', 'test/' subdirectories
        batch_size: Batch size for training
        num_workers: Parallel data loading workers
    
    Returns:
        dict with 'train', 'val', 'test' DataLoaders and class_weights
    """
    datasets = {
        'train': ChestXrayDataset(os.path.join(data_root, 'train'), transform=train_transform),
        'val':   ChestXrayDataset(os.path.join(data_root, 'val'),   transform=val_transform),
        'test':  ChestXrayDataset(os.path.join(data_root, 'test'),  transform=val_transform),
    }
    
    dataloaders = {
        split: DataLoader(
            ds,
            batch_size=batch_size,
            shuffle=(split == 'train'),
            num_workers=num_workers,
            pin_memory=torch.cuda.is_available()
        )
        for split, ds in datasets.items()
    }
    
    class_weights = datasets['train'].get_class_weights()
    
    print('\n=== DataLoader Summary ===')
    for split, ds in datasets.items():
        print(f'{split:>5}: {len(ds):>5} images')
    print(f'\nClass weights (for balanced loss): {class_weights.tolist()}')
    
    return dataloaders, class_weights


# Verify with actual data if available
DATA_ROOT = '../data'
if os.path.exists(DATA_ROOT):
    loaders, weights = get_dataloaders(DATA_ROOT)
    # Test one batch
    images, labels = next(iter(loaders['train']))
    print(f'\nBatch shape: {images.shape}')   # expect (32, 3, 224, 224)
    print(f'Label range: {labels.min()}–{labels.max()}')
else:
    print('Data directory not found. Download datasets first.')
    print('Expected structure: data/train/{Normal,COVID-19,Viral Pneumonia}/')

## 4. Visualise Augmented Samples

In [ ]:
def denormalise(tensor):
    """Reverse ImageNet normalisation for display."""
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return torch.clamp(tensor * std + mean, 0, 1)


def show_augmented_samples(dataset, n_per_class=4):
    """Show original vs augmented images side by side."""
    classes = ChestXrayDataset.CLASSES
    fig, axes = plt.subplots(len(classes), n_per_class + 1,
                              figsize=((n_per_class + 1) * 3, len(classes) * 3))
    
    class_colors = {'Normal': '#2ecc71', 'COVID-19': '#e74c3c', 'Viral Pneumonia': '#f39c12'}
    
    for row, cls in enumerate(classes):
        # Find one sample path
        sample_path = None
        for path, label in dataset.samples:
            if label == ChestXrayDataset.CLASS2IDX[cls]:
                sample_path = path
                break
        
        if sample_path is None:
            continue
        
        # Original
        orig = Image.open(sample_path).convert('RGB')
        ax = axes[row, 0]
        ax.imshow(orig)
        ax.set_title('Original', fontsize=9)
        ax.set_ylabel(cls, fontsize=11, fontweight='bold',
                      color=class_colors.get(cls, 'black'))
        ax.axis('off')
        
        # Augmented
        for col in range(1, n_per_class + 1):
            aug = train_transform(orig)
            img = denormalise(aug).permute(1, 2, 0).numpy()
            ax = axes[row, col]
            ax.imshow(img)
            ax.set_title(f'Aug {col}', fontsize=9)
            ax.axis('off')
    
    plt.suptitle('Data Augmentation Examples (Train Set Only)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../reports/figures/augmentation_examples.png', bbox_inches='tight', dpi=150)
    plt.show()


if os.path.exists(DATA_ROOT):
    train_ds = ChestXrayDataset(os.path.join(DATA_ROOT, 'train'), transform=None)
    show_augmented_samples(train_ds)
else:
    print('Data not available — augmentation pipeline defined and ready.')

## 5. Stratified Split Helper

If the dataset is not pre-split, use this helper to create the 70/15/15 stratified split.

In [ ]:
def create_stratified_split(source_root, dest_root, val_ratio=0.15, test_ratio=0.15, seed=42):
    """
    Split a flat class-organised dataset into train/val/test.
    
    Expected source structure:
        source_root/
          Normal/ ... .jpg
          COVID-19/ ... .jpg
          Viral Pneumonia/ ... .jpg
    
    Creates:
        dest_root/
          train/ val/ test/  (each with class subdirectories)
    """
    np.random.seed(seed)
    classes = ['Normal', 'COVID-19', 'Viral Pneumonia']
    
    for cls in classes:
        src_cls = os.path.join(source_root, cls)
        if not os.path.exists(src_cls):
            print(f'  Skipping {cls} — directory not found')
            continue
        
        files = [f for f in os.listdir(src_cls)
                 if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        
        # Stratified split
        train_files, temp_files = train_test_split(
            files, test_size=(val_ratio + test_ratio), random_state=seed, shuffle=True
        )
        val_files, test_files = train_test_split(
            temp_files, test_size=test_ratio/(val_ratio + test_ratio), random_state=seed
        )
        
        for split, file_list in [('train', train_files), ('val', val_files), ('test', test_files)]:
            split_cls_dir = os.path.join(dest_root, split, cls)
            os.makedirs(split_cls_dir, exist_ok=True)
            for fname in file_list:
                shutil.copy2(os.path.join(src_cls, fname),
                             os.path.join(split_cls_dir, fname))
        
        print(f'{cls}: {len(train_files)} train | {len(val_files)} val | {len(test_files)} test')
    
    print('\nSplit complete. Run Notebook 03 to start training.')


# Uncomment to run:
# create_stratified_split(source_root='../data/raw', dest_root='../data')
print('Split function defined. Uncomment last line to execute.')

## 6. Summary

| Step | Detail |
|---|---|
| Input size | 224×224 (ViT-B/16 requirement) |
| Channels | 3-channel RGB (greyscale replicated) |
| Normalisation | ImageNet mean/std |
| Train augmentation | Flip, rotate ±15°, colour jitter, random crop |
| Split | 70% train / 15% val / 15% test (stratified) |
| Imbalance handling | Class-weighted loss in training |

**Next: Notebook 03 — Model Training**